# B: Reviewers A and B (prompt tuning and evaluation)

The reviewers are **not trained**. They are Llama with one prompt (`prompts/reviewer_*.txt`), where only the
checked side changes between Reviewer A and Reviewer B. "Training" here means tuning that prompt against a
small human-labeled set, `data/reviewer_cases.csv`:

- 40 cases = 20 **mirror pairs** (the same bias with the sides swapped), each labeled `unfair_to` =
  `none` / `palestinian` / `israeli` / `ukrainian` / `russian`.
- **dev** (10 pairs): tune on these as often as you like.
- **heldout** (10 pairs): score **once**, at the end. That is the number for the report. Don't read these cases while tuning.

The loop: change ONE thing in the prompt -> run the dev cell -> compare with `history()` -> keep or revert.
Every run is logged with the prompt version (a hash of the prompt files + guidebook rubric) to
`ARTIFACTS/results/reviewer_runs.jsonl`. Commit every prompt change so a version can be traced to its text.

Needs: T4 GPU and `HF_TOKEN` with Llama 3.1 access (otherwise Qwen2.5-7B is used; note which one in the report).

In [ ]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:  # autoreload is broken on some Colab images (Python 3.13: no 'imp' module)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload not available: restart the runtime after editing src/")

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

## 1. The labeled cases

In [ ]:
import pandas as pd
from src import reviewers, reviewer_eval, rewriter

cases = reviewer_eval.load_cases()
print(pd.crosstab(cases["split"], cases["unfair_to"]))
print("prompt version:", reviewers.prompt_version())
reviewer_eval.load_cases("dev")[["case_id", "unfair_to", "bias_type", "revised"]]

## 2. Load Llama once (shared with the rewriter)

In [ ]:
import time, torch
start = time.perf_counter()
_, _, model_id = rewriter.get_model()
print(f"{model_id} loaded in {time.perf_counter() - start:.0f} s")
if torch.cuda.is_available():
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. One case, both reviewers (look at the raw answers)

In [ ]:
case = cases.set_index("case_id").loc["P02a"]
print("ORIGINAL:", case["original"]); print("REVISED: ", case["revised"]); print("LABEL: unfair to", case["unfair_to"])
for p in config.PERSPECTIVES[case["conflict"]]:
    res, info = reviewers.review_detailed(case["original"], case["revised"], p)
    print(f"\n[{p}] fair={res['fair']}  issues={res['issues']}  retries={info['retries']}  fell_back={info['fell_back']}")
    print("raw:", info["attempts"][-1]["raw"][:400])

## 4. Dev run (repeat after every prompt change)

Write in `NOTE` what you changed, e.g. `"v2: added example for R09 (unequal verbs)"`.

In [ ]:
NOTE = "v1: first prompt (rubric from guidebook + 3 Side X/Side Y examples)"

start = time.perf_counter()
calls, m = reviewer_eval.run("dev", NOTE)
print(f"{len(calls)} reviewer calls in {time.perf_counter() - start:.0f} s")
print(pd.Series(m["summary"]).to_string())
print("per reviewer accuracy:", m["per_perspective"])

In [ ]:
# The mistakes: what did the reviewer say, and what should it have said?
pd.set_option("display.max_colwidth", 100)
wrong = calls[~calls["correct"]]
wrong[["case_id", "perspective", "unfair_to", "fair", "phrases", "problems", "retries", "fell_back"]]

In [ ]:
# Every version so far (dev and held-out)
reviewer_eval.history()

## How to read the dev numbers

- **unfair_caught** low: the reviewer misses bias against its side. Add or sharpen an example for the missed bias type.
- **false_alarm_rate** high: it complains about fair rewrites (e.g. about facts that look bad for its side). Sharpen the "It is NOT unfair" part.
- **symmetry** < 1: Reviewer A and B judge mirror cases differently. Look at the pair in `wrong`; this matters most for the report.
- **fallback_rate** > 0: the JSON could not be read; look at `raw` in `calls`.
- **agreement** is *not* supposed to be 1: on an unfair case only the harmed side's reviewer should complain.

Change one thing, commit the prompt, run section 4 again.

## 5. Held-out: run ONCE, when tuning is finished

Set `RUN_HELDOUT = True` only for the final score. If it was already run, report the first run (the log keeps all of them).

In [ ]:
RUN_HELDOUT = False

if RUN_HELDOUT:
    calls_h, m_h = reviewer_eval.run("heldout", "final: prompt " + reviewers.prompt_version(), final=True)
    print(pd.Series(m_h["summary"]).to_string())
    print("per reviewer accuracy:", m_h["per_perspective"])